# Maia Lite 0.3.0 — Chinchilla / Corpus v8 Token Budget Audit

**Read-only audit** for the 507,026,432-parameter model and A100 80 GB. Computes the *actual token counts already recorded by the production pipeline*, reconciles complete 2,048-token blocks against pending tails, and derives the Chinchilla budget. Does **not** start training, modify SQLite, reprocess source chunks, or approve licenses.

**Crucial distinctions:** `completed_chunk.tokens` counts tokenized, accepted, deduplicated documents (plus EOS) across all splits; `shard.blocks × 2048` counts persisted full blocks; `pending` holds incomplete tails; only the **train split** can feed training. Processed does **not** mean legally eligible. The remaining 650 Wikipedia PT chunks are not included in completed-token totals.

In [ ]:
from pathlib import Path
import json, sqlite3, hashlib, math, os, collections, datetime
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    print('Not running in Colab: Drive must be accessible at the configured path.')

BASE = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_0_3_0_Chat/corpus_v8/runs/acquisition_v8_5_001')
OUT = BASE/'pretraining_production_v1_full'
RECOVERY = OUT/'recovery_audit_v4'
CONFIG = OUT/'run_config.json'
DB_CANDIDATES = [RECOVERY/'production_resume_v2/state_resume_v2.sqlite3', RECOVERY/'state_operational_v5.sqlite3']
DB = next((p for p in DB_CANDIDATES if p.is_file()), None)
assert DB is not None, f'No known production DB found: {DB_CANDIDATES}'
assert CONFIG.is_file(), f'Missing immutable production configuration: {CONFIG}'
print('Using SQLite (read-only):', DB)
print('Run configuration:', CONFIG)


## 1. Immutable inventory and tokenizer fingerprint

Use the tokenizer already validated by the corpus pipeline. Do **not** substitute another tokenizer (even if it has the same vocabulary size). The notebook verifies file fingerprints from the production configuration, without requiring a full retokenization.

In [ ]:
config = json.loads(CONFIG.read_text(encoding='utf-8'))
assert config.get('schema_version') == '1.1-full', 'Unexpected production configuration version'
assert config.get('block_size') == 2048
inventory = config['selected_inventory']
assert len(inventory) == 21021, f'Unexpected inventory size: {len(inventory)}'
INV_SHA = hashlib.sha256(json.dumps(inventory, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode()).hexdigest()
EXPECTED_INV_SHA = 'ad19d1d98e2cf5b91e650f343e6430bbde34b549b5fbc352b47682a91d7ab562'
assert INV_SHA == EXPECTED_INV_SHA, 'Frozen inventory differs from audited inventory'
TOKENIZER_DIR = Path('/content/drive/MyDrive/Colab_LLMs/Maia_Lite_v3/tokenizer')
expected = config.get('tokenizer_file_sha256', {})
assert expected, 'Missing tokenizer fingerprint in run_config.json'
actual = {}
for name, sha in expected.items():
    path = TOKENIZER_DIR/name
    assert path.is_file(), f'Missing tokenizer file: {path}'
    actual[name] = hashlib.sha256(path.read_bytes()).hexdigest()
assert actual == expected, 'Tokenizer differs from production tokenizer; do not mix tokenizations'
print('Inventory SHA-256:', INV_SHA)
print('Tokenizer file SHA-256 verified:', len(actual), 'files')
print('Source inventory:', dict(collections.Counter(r['source'] for r in inventory)))


## 2. Read-only SQL audit: processed tokens, persisted blocks and split breakdown

The `completed_chunk.tokens` total is authoritative for accepted tokenized documents. For each source, the difference from `shard.blocks × 2048 + pending bytes/4` should be zero (assuming all committed shards and tails are represented). Note that SQLite row counts and the underlying shard-file contents are **not** rehashed here; this is a fast accounting audit, not a full integrity scan.

In [ ]:
uri = DB.resolve().as_uri() + '?mode=ro'
con = sqlite3.connect(uri, uri=True, timeout=30)
try:
    assert con.execute('PRAGMA quick_check').fetchone()[0] == 'ok'
    completed = con.execute('SELECT source,COUNT(*),COALESCE(SUM(docs),0),COALESCE(SUM(duplicates),0),COALESCE(SUM(tokens),0) FROM completed_chunk GROUP BY source').fetchall()
    shards = con.execute('SELECT source,split,COUNT(*),COALESCE(SUM(blocks),0) FROM shard GROUP BY source,split').fetchall()
    tails = con.execute('SELECT source,split,length(ids) FROM pending').fetchall()
finally:
    con.close()
by_src = {s: {'chunks':c,'docs':d,'duplicates':dup,'tokens':t} for s,c,d,dup,t in completed}
block_by = {(s,sp): blocks for s,sp,n,blocks in shards}
shard_count_by = {(s,sp): n for s,sp,n,blocks in shards}
tail_by = {(s,sp): (nbytes or 0)//4 for s,sp,nbytes in tails}
assert all((nbytes or 0)%4==0 for _,_,nbytes in tails), 'Invalid pending uint32 length'
rows=[]
for source in sorted(set(r['source'] for r in inventory)):
    chunk_total = sum(1 for r in inventory if r['source']==source)
    item = by_src.get(source, {'chunks':0,'docs':0,'duplicates':0,'tokens':0})
    split_counts = {sp: block_by.get((source,sp),0)*2048 + tail_by.get((source,sp),0) for sp in ('train','validation','test')}
    accounted = sum(split_counts.values())
    delta = item['tokens']-accounted
    rows.append({'source':source,'inventory_chunks':chunk_total,'completed_chunks':item['chunks'],
                 'remaining_chunks':chunk_total-item['chunks'],'accepted_documents':item['docs'],
                 'duplicate_documents':item['duplicates'],'tokenized_total':item['tokens'],
                 'train_tokens':split_counts['train'],'validation_tokens':split_counts['validation'],
                 'test_tokens':split_counts['test'],'accounting_delta':delta,
                 'train_full_blocks':block_by.get((source,'train'),0),
                 'train_pending_tokens':tail_by.get((source,'train'),0)})
for r in rows:
    print(f"{r['source']:16} chunks {r['completed_chunks']:5}/{r['inventory_chunks']:5} | tokens {r['tokenized_total']:>14,} | train {r['train_tokens']:>14,} | delta {r['accounting_delta']:+,}")
assert all(r['accounting_delta']==0 for r in rows), 'Token accounting mismatch: inspect DB and source streams before proceeding'
assert sum(r['completed_chunks'] for r in rows) <= len(inventory)
print('TOTAL completed:', sum(r['completed_chunks'] for r in rows),'/',len(inventory))
print('TOTAL remaining:', sum(r['remaining_chunks'] for r in rows))


## 3. Chinchilla budget and language mix

20 tokens per parameter is a planning heuristic for compute-efficient training, not a legal clearance or guaranteed optimum. The available **train** tokens are counted from existing materialized streams. A completed stream may contain rights-pending material: **do not use these totals as an eligibility count**. PT/EN/ES shares reflect materialized train tokens, not the desired sampling proportions.

In [ ]:
PARAMS = 507_026_432
RATIO = 20
TARGET = PARAMS*RATIO
MICROBATCH=8; ACCUM=4; CONTEXT=2048
TOKENS_PER_UPDATE=MICROBATCH*ACCUM*CONTEXT
STEPS = math.ceil(TARGET/TOKENS_PER_UPDATE)
lang={'fineweb2_es':'es','fineweb2_pt':'pt','fineweb_edu_en':'en','wikipedia_en':'en','wikipedia_es':'es','wikipedia_pt':'pt'}
train_total = sum(r['train_tokens'] for r in rows)
materialized_total = sum(r['tokenized_total'] for r in rows)
train_full = sum(r['train_full_blocks']*2048 for r in rows)
per_language=collections.Counter()
for r in rows: per_language[lang[r['source']]] += r['train_tokens']
print(f'Parameters: {PARAMS:,}')
print(f'Chinchilla target: {TARGET:,} tokens (20x)')
print(f'Tokens/update: {TOKENS_PER_UPDATE:,}; planned updates: {STEPS:,}')
print(f'Processed accepted tokens (all splits): {materialized_total:,}')
print(f'Materialized train tokens (incl. pending tails): {train_total:,}')
print(f'Train tokens in complete blocks: {train_full:,}')
print(f'Chinchilla target / current materialized train tokens: {TARGET/train_total:.3f}x' if train_total else 'No materialized train tokens')
for language in ('pt','en','es'):
    count=per_language[language]
    print(f'{language}: {count:,} train tokens ({count/train_total:.2%})' if train_total else f'{language}: 0')
print('NOTE: These are NOT approved-for-training totals.')


## 4. Rights gate and reproducible planning report

Writes **only** a new report under `/content/maia_chinchilla_audit` (local Colab runtime), not to the corpus Drive. The report records the license gate as unresolved. It does not set `eligible_for_training`, create approval files, or run training.

In [ ]:
PROVENANCE = RECOVERY/'production_resume_v2/wikimedia_rights_review/provenance_audit/provenance_summary.json'
provenance = json.loads(PROVENANCE.read_text(encoding='utf-8')) if PROVENANCE.is_file() else None
if provenance:
    assert provenance.get('inventory_sha256') == INV_SHA
    assert provenance.get('scanned_chunks') == 650 and provenance.get('total_remaining_chunks') == 650
    print('Wikipedia PT provenance:', provenance.get('status'))
else:
    print('WARNING: provenance report not found; review manually')
RIGHTS_APPROVAL = RECOVERY/'production_resume_v2/license_approval.json'
print('License approval file exists:', RIGHTS_APPROVAL.is_file())
print('This notebook does not interpret mere existence as valid authorization.')
report={
 'schema_version':1,
 'generated_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
 'model':{'name':'Maia Lite 0.3.0','parameters':PARAMS,'training_from_scratch':True,'context':CONTEXT},
 'tokenizer_file_sha256':actual,'inventory_sha256':INV_SHA,
 'production_db_path':str(DB),'target_ratio_tokens_per_parameter':RATIO,
 'target_train_token_exposures':TARGET,'microbatch':MICROBATCH,'gradient_accumulation':ACCUM,
 'tokens_per_optimizer_update':TOKENS_PER_UPDATE,'planned_optimizer_updates':STEPS,
 'processed_accepted_tokens_all_splits':materialized_total,
 'materialized_train_tokens_including_tails':train_total,
 'materialized_train_tokens_in_full_blocks':train_full,
 'language_train_token_counts':dict(per_language),'sources':rows,
 'remaining_chunks':sum(r['remaining_chunks'] for r in rows),
 'wikipedia_pt_provenance_status':provenance.get('status') if provenance else None,
 'rights_review':'PENDING_SOURCE_SPECIFIC_REVIEW',
 'eligible_training_tokens':None,
 'production_training_authorized':False,
 'limitations':['Committed token counts are not independent text tokens or rights-cleared tokens.',
                'Unprocessed chunks are excluded from counts.',
                'Source-specific rights must be assessed before using any material in training.',
                'No tokenizer-based rescan or shard content checksum is performed.']
}
REPORT_DIR=Path('/content/maia_chinchilla_audit');REPORT_DIR.mkdir(parents=True,exist_ok=True)
REPORT_PATH=REPORT_DIR/'maia_v8_chinchilla_token_budget.json'
REPORT_PATH.write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print('Report:',REPORT_PATH)
print('Production training authorized:',report['production_training_authorized'])


## 5. Interpretation and next gate

- Compare `train_full_blocks` against the intended training data loader; the loader should read complete uint32 blocks, never the incomplete tails.
- After remaining chunks are **legitimately authorized and processed**, rerun this notebook to get final exact token counts.
- Validate that the tokenizer fingerprint and model vocabulary match; confirm source-specific rights, source mixture, train/validation/test isolation, and checkpoint-resume semantics before real training.
- The Chinchilla budget is **tokens consumed**. If the train corpus is smaller, repeating samples increases exposures but does not increase independent data. Report both quantities.
- For a production trainer, sample across sources by an explicit policy; do not silently infer a 40/35/25 language mixture from the current counts.